#### Environment Check


In [1]:
import sys
print(sys.executable)


/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python


#### Setup


In [3]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field


#### Project Paths


In [4]:
cwd = Path.cwd()

if (cwd / "code").exists() and (cwd / "data").exists():
    LAB_DIR = cwd
elif (cwd.parent / "code").exists() and (cwd.parent / "data").exists():
    LAB_DIR = cwd.parent
else:
    LAB_DIR = Path("..").resolve()

CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"
REPORTS_DIR = LAB_DIR / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

str(LAB_DIR)


'/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab'

#### Import Course Helpers


In [5]:
import sys

sys.path.append(str(CODE_DIR))

from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress


#### Load OpenAI Client


In [6]:
PROJECT_ROOT = LAB_DIR.parents[1]
ENV_PATH = PROJECT_ROOT / ".env"
loaded = load_dotenv(ENV_PATH)
print("Env file:", ENV_PATH)
print("Env loaded:", loaded)
openai_client = OpenAI()
MODEL = "gpt-5.4-mini"


Env file: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.env
Env loaded: True


#### Load RAG Answers


In [7]:
rags_path = DATA_DIR / "rag-answers-new.csv"

df_answers = pd.read_csv(rags_path)
answers = df_answers.to_dict(orient="records")

len(answers)


565

#### Define Judge Output


In [8]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )


#### Judge Instructions


In [9]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()


#### Test Judge Prompt


In [10]:
rec = answers[0]

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"],
)

print(prompt)


Question:
How do the lectures, notebooks, and homework usually fit together during a typical week?

Original Answer (ground truth):
Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

A typical workflow is:

1. Watch the lesson videos.
2. Work through the lesson notebooks/code.
3. Read the homework instructions on GitHub.
4. Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

AI Answer:
A typical weekly workflow is:

1. Watch the lesson videos.
2. Work through the l

#### Judge One Answer


In [11]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
    model=MODEL,
)

eval_result


AnswerEvaluation(reasoning='The AI answer captures the core workflow exactly: watch lectures, then work through notebooks/code, then read homework instructions, and submit on the course platform. This matches the ground truth’s typical week structure. It omits the extra introductory links and the note that homework may use a different dataset or slightly different task, but those are not essential to the question asked.', score='good')

#### Check One Judge Cost


In [12]:
calc_price(usage)


0.0001305

#### Create Judge Function


In [13]:
def evaluate_aqa(question, answer_orig, answer_llm, model=MODEL):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm,
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage


#### Create Record Judge Function


In [14]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"],
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_orig": rec["answer_orig"],
        "answer_llm": rec["answer_llm"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage


#### Run LLM Judge


In [15]:
answers_to_judge = answers
# answers_to_judge = answers[:20]

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers_to_judge, judge_record)


  0%|          | 0/565 [00:00<?, ?it/s]

#### Split Evaluations And Usage


In [16]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

len(evaluations)


565

#### Create Evaluation DataFrame


In [17]:
df_eval = pd.DataFrame(evaluations)

df_eval.head()


,question,document,answer_orig,answer_llm,score,reasoning
0,"I signed up for the LLM Zoomcamp, but I still ...",977bf7786c,You don't need it. You're accepted. You can al...,No need to worry — you don’t need a confirmati...,good,The AI answer conveys the same key points as t...
1,Where can I find the deadlines and submit my h...,04919992b3,Start with the [LLM Zoomcamp docs](https://dat...,You can find the deadlines and submit your hom...,good,The AI answer correctly identifies the course ...
2,"How do the lectures, notebooks, and homework u...",04919992b3,Start with the [LLM Zoomcamp docs](https://dat...,A typical weekly workflow is:\n\n1. Watch the ...,good,The AI answer matches the core workflow from t...
3,What is the recommended way to begin the cours...,04919992b3,Start with the [LLM Zoomcamp docs](https://dat...,Start with the following:\n\n1. The [LLM Zoomc...,good,The AI answer matches the ground truth: it rec...
4,Which course resources should I open first if ...,04919992b3,Start with the [LLM Zoomcamp docs](https://dat...,Start with these resources in this order:\n\n1...,good,The AI answer captures the key resources to op...


#### Check Judge Scores


In [18]:
score_counts = df_eval["score"].value_counts()
score_rate = df_eval["score"].value_counts(normalize=True)

score_counts, score_rate


(score
 good    528
 bad      37
 Name: count, dtype: int64,
 score
 good    0.934513
 bad     0.065487
 Name: proportion, dtype: float64)

#### Calculate Judge Cost


In [19]:
total_cost = calc_total_price(usages)

total_cost


0.06600195

#### Inspect Bad Answers


In [20]:
df_eval[df_eval["score"] == "bad"].head()


,question,document,answer_orig,answer_llm,score,reasoning
17,What is the deadline condition for getting a c...,74eb249bbf,"Yes, but if you want to receive a certificate,...",I don't know.,bad,The ground truth says late signups can still g...
18,Can I enroll after the course has already begu...,74eb249bbf,"Yes, but if you want to receive a certificate,...",You can start whenever you want — joining is n...,bad,The AI answer captures that enrollment is poss...
19,"If I’m joining the course late, will I still b...",74eb249bbf,"Yes, but if you want to receive a certificate,...",I don’t know.,bad,The AI answer does not convey the ground truth...
27,Do you know the upcoming session date for this...,bd31146b0e,Summer 2027.,I don't know.,bad,The ground truth states the upcoming session d...
34,Can you tell me when the course is scheduled t...,bd31146b0e,Summer 2027.,I don't know.,bad,The ground truth gives a specific return time:...


#### Save Judge Results


In [21]:
output_path = DATA_DIR / "rag-evaluations-new.csv"
df_eval.to_csv(output_path, index=False)

output_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data/rag-evaluations-new.csv')

#### Save Judge Report


In [22]:
good_count = int((df_eval["score"] == "good").sum())
bad_count = int((df_eval["score"] == "bad").sum())
total_count = len(df_eval)

good_rate = good_count / total_count

report_path = REPORTS_DIR / "rag_judge_metrics.md"

report_lines = [
    "# RAG LLM-as-a-Judge Metrics",
    "",
    f"- Total answers judged: {total_count}",
    f"- Good answers: {good_count}",
    f"- Bad answers: {bad_count}",
    f"- Good rate: {good_rate:.2%}",
    f"- Judge cost: {total_cost}",
    f"- Output file: {output_path.name}",
]

with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
    f.write("\n")

report_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/reports/rag_judge_metrics.md')